# 04 session attribution

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = PUBLICATION_ROOT
DATA_DIR = PROJECT_ROOT / "data" / "ground_truth"

ledger = pd.read_csv(DATA_DIR / "execution_ledger.csv")

# Convert timestamps to UTC datetime objects
ledger["ScheduledUTC"] = pd.to_datetime(
    ledger["ScheduledUTC"], utc=True
)

ledger["ExecutionUTC"] = pd.to_datetime(
    ledger["ExecutionUTC"], utc=True
)

# Examine the execution history of the first session
first_session_id = ledger["SessionID"].iloc[0]

display(
    ledger.loc[
        ledger["SessionID"] == first_session_id,
        ["SessionID", "ScheduledUTC", "ExecutionUTC", "Status"]
    ]
)

In [ ]:
# Extract the start time of each session
session_starts = (
    ledger[ledger["Status"] == "Started"]
    .groupby("SessionID")["ExecutionUTC"]
    .min()
    .rename("StartUTC")
)

# Extract the final execution time of each session
session_ends = (
    ledger[ledger["Status"].isin(["Completed", "Failed"])]
    .groupby("SessionID")["ExecutionUTC"]
    .max()
    .rename("EndUTC")
)

# Combine into one table
session_windows = pd.concat(
    [session_starts, session_ends],
    axis=1
).reset_index()

# Calculate session duration
session_windows["DurationSeconds"] = (
    session_windows["EndUTC"] - session_windows["StartUTC"]
).dt.total_seconds()

print("Sessions with execution windows:", len(session_windows))

display(session_windows.head())

In [ ]:
# Summarise session durations
print("Session duration statistics (seconds):")
display(
    session_windows["DurationSeconds"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

# Identify sessions lasting longer than 10 minutes
long_sessions = session_windows[
    session_windows["DurationSeconds"] > 600
].copy()

print("\nSessions longer than 10 minutes:", len(long_sessions))

display(
    long_sessions.sort_values(
        "DurationSeconds", ascending=False
    ).head(10)
)

In [ ]:
# Examine execution history for unusually long sessions
long_session_ids = long_sessions["SessionID"]

long_history = ledger[
    ledger["SessionID"].isin(long_session_ids)
].copy()

# Sort events chronologically
long_history = long_history.sort_values(
    ["SessionID", "ExecutionUTC"]
)

# Show the status history for the five longest sessions
top_five_ids = (
    long_sessions
    .sort_values("DurationSeconds", ascending=False)
    .head(5)["SessionID"]
)

display(
    long_history[
        long_history["SessionID"].isin(top_five_ids)
    ][
        ["SessionID", "ExecutionUTC", "Status", "Detail"]
    ]
)

In [ ]:
# Reconstruct session windows using the actual execution evidence

from datetime import timedelta

# Load the ground-truth file generated in Notebook 02
ground_truth = pd.read_csv(
    PROJECT_ROOT / "data" / "notebook_outputs" / "session_ground_truth.csv"
)

# Load M05 worker results from their original JSON files
import json

m05_dir = DATA_DIR / "M05Results"
m05_results = []

for file_path in m05_dir.glob("*.json"):
    with open(file_path, "r", encoding="utf-8-sig") as file:
        result = json.load(file)

    m05_results.append({
        "SessionID": result.get("SessionID"),
        "WorkerStartUTC": result.get("StartedUTC"),
        "WorkerEndUTC": result.get("EndedUTC")
    })

m05_df = pd.DataFrame(m05_results)

# Attach M05 worker timestamps to scheduled sessions
ground_truth = ground_truth.merge(
    m05_df,
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Preserve the original order of the execution ledger
ledger_history = {
    session_id: group
    for session_id, group in ledger.groupby("SessionID", sort=False)
}

corrected_rows = []

for _, session in ground_truth.iterrows():

    sid = session["SessionID"]
    history = ledger_history.get(sid)

    start = pd.NaT
    end = pd.NaT
    source = ""
    issue = ""

    try:
        if history is None:
            raise ValueError("Missing ledger history")

        if session["ScenarioID"] == "M05":
            start = pd.to_datetime(session["WorkerStartUTC"], utc=True)
            end = pd.to_datetime(session["WorkerEndUTC"], utc=True)
            source = "worker_interval"

        else:
            started = history[history["Status"] == "Started"]

            if len(started) != 1:
                raise ValueError("Expected exactly one Started record")

            start_index = started.index[0]
            start = ledger.loc[start_index, "ExecutionUTC"]

            # Find the first handler return after Started
            later = ledger.loc[
                history.index[history.index > start_index]
            ]

            returned = later[
                later["Status"].isin(
                    ["Completed", "Failed", "InvokedUnverified"]
                )
                & ~later["Detail"].fillna("").str.startswith("Reconciled:")
            ]

            if returned.empty:
                raise ValueError("Missing immediate handler return")

            end = returned.iloc[0]["ExecutionUTC"]
            source = "ledger_start_to_handler_return"

        if pd.isna(start) or pd.isna(end) or end < start:
            raise ValueError("Invalid execution interval")

    except (ValueError, TypeError, KeyError) as error:
        start = pd.NaT
        end = pd.NaT
        issue = str(error)

    corrected_rows.append({
        "SessionID": sid,
        "StartUTC": start,
        "EndUTC": end,
        "WindowSource": source,
        "WindowIssue": issue
    })

corrected_windows = pd.DataFrame(corrected_rows)

corrected_windows["DurationSeconds"] = (
    corrected_windows["EndUTC"] - corrected_windows["StartUTC"]
).dt.total_seconds()

print("Total sessions:", len(corrected_windows))
print("Valid windows:", corrected_windows["WindowIssue"].eq("").sum())
print("Unverified windows:", corrected_windows["WindowIssue"].ne("").sum())

print("\nCorrected duration statistics:")
display(corrected_windows["DurationSeconds"].describe())

print("\nLongest corrected windows:")
display(
    corrected_windows.sort_values(
        "DurationSeconds", ascending=False
    ).head(10)
)

In [ ]:
# Apply the same 2-second padding 
PADDING_SECONDS = 2

corrected_windows["ObservationStartUTC"] = (
    corrected_windows["StartUTC"]
    - pd.Timedelta(seconds=PADDING_SECONDS)
)

corrected_windows["ObservationEndUTC"] = (
    corrected_windows["EndUTC"]
    + pd.Timedelta(seconds=PADDING_SECONDS)
)

# The prediction cutoff is the observation end
corrected_windows["PredictionCutoffUTC"] = (
    corrected_windows["ObservationEndUTC"]
)

display(
    corrected_windows[
        [
            "SessionID",
            "WindowSource",
            "ObservationStartUTC",
            "ObservationEndUTC",
            "PredictionCutoffUTC"
        ]
    ].head()
)

In [ ]:
# Attach scheduled usernames to the observation windows
session_users = ground_truth[
    ["SessionID", "Username", "ScenarioID"]
].copy()

attribution_windows = corrected_windows.merge(
    session_users,
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Normalise usernames for matching
attribution_windows["NormalizedUsername"] = (
    attribution_windows["Username"]
    .str.lower()
    .str.split("\\", regex=False)
    .str[-1]
)

print("Session windows:", len(attribution_windows))

display(
    attribution_windows[
        [
            "SessionID",
            "Username",
            "NormalizedUsername",
            "ScenarioID",
            "ObservationStartUTC",
            "PredictionCutoffUTC"
        ]
    ].head()
)

In [ ]:
# Examine usernames in actual Windows security events

EVENTS_FILE = (
    PROJECT_ROOT / "data" / "notebook_outputs"
    / "wazuh_structured_events.csv"
)

sample = pd.read_csv(
    EVENTS_FILE,
    nrows=100000,
    dtype=str
)

# Look at events with identifiable users
user_events = sample[
    sample["Username"].notna() |
    sample["TargetUsername"].notna()
]

display(
    user_events[
        [
            "EventID",
            "Provider",
            "Username",
            "TargetUsername",
            "EventChannel"
        ]
    ].head(15)
)

In [ ]:
# Read structured Wazuh telemetry and prepare event-specific user identities

import csv
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs"

EVENTS_FILE = OUTPUT_DIR / "wazuh_structured_events.csv"
identity_file = OUTPUT_DIR / "wazuh_identity_events.csv"

# Authentication events identify the target account
target_account_events = {"4624", "4625", "4776"}

event_count = 0

with open(identity_file, "w", newline="", encoding="utf-8") as output:
    writer = None

    for chunk in pd.read_csv(
        EVENTS_FILE,
        chunksize=50000,
        dtype=str
    ):

        # Authentication events use the target account.
        # Other Security events use the subject account.
        # Sysmon events use their User field.
        is_target = (
            chunk["Provider"].eq("Microsoft-Windows-Security-Auditing")
            & chunk["EventID"].isin(target_account_events)
        )

        # Select the appropriate username
        chunk["EventUsername"] = chunk["Username"].where(
            ~is_target,
            chunk["TargetUsername"]
        )

        # Select the appropriate Windows domain
        chunk["EventDomain"] = chunk["SubjectDomainName"].where(
            ~is_target,
            chunk["TargetDomainName"]
        )

        # Sysmon usernames may already contain the domain
        chunk["EventIdentity"] = chunk["EventUsername"].fillna("")

        # Add domain where the identity is not already qualified
        has_domain = (
            chunk["EventDomain"].notna()
            & chunk["EventDomain"].ne("")
            & chunk["EventIdentity"].ne("")
            & ~chunk["EventIdentity"].str.contains("\\", regex=False)
        )

        chunk.loc[has_domain, "EventIdentity"] = (
            chunk.loc[has_domain, "EventDomain"]
            + "\\"
            + chunk.loc[has_domain, "EventIdentity"]
        )

        # Normalise Windows account identities
        # Collapse repeated backslashes and convert to lowercase
        chunk["EventIdentity"] = (
            chunk["EventIdentity"]
            .str.strip()
            .str.casefold()
            .str.replace(r"\\+", r"\\", regex=True)
        )

        # Extract the username without the domain
        chunk["NormalizedEventUsername"] = (
            chunk["EventIdentity"]
            .str.split("\\", regex=False)
            .str[-1]
        )

        # Initialise CSV writer using all available columns
        if writer is None:
            writer = csv.DictWriter(
                output,
                fieldnames=chunk.columns.tolist()
            )
            writer.writeheader()

        # Write the processed events
        writer.writerows(
            chunk.where(pd.notna(chunk), None).to_dict("records")
        )

        event_count += len(chunk)

print("Identity enrichment completed.")
print("Events processed:", event_count)
print("Saved to:", identity_file)

In [ ]:
# Identify events mentioning simulated usernames
simulation_users = set(
    attribution_windows["NormalizedUsername"].dropna()
)

total_events = 0
matching_user_events = 0
matching_by_type = {}

for chunk in pd.read_csv(
    identity_file,
    chunksize=50000,
    dtype=str,
    usecols=["EventID", "NormalizedEventUsername"]
):
    matches = chunk["NormalizedEventUsername"].isin(simulation_users)

    total_events += len(chunk)
    matching_user_events += matches.sum()

    counts = chunk.loc[matches, "EventID"].value_counts()

    for event_id, count in counts.items():
        matching_by_type[event_id] = (
            matching_by_type.get(event_id, 0) + count
        )

print("Total events:", total_events)
print("Events with matching username:", matching_user_events)

display(
    pd.Series(matching_by_type)
    .sort_values(ascending=False)
    .rename("EventCount")
    .to_frame()
)

In [ ]:
# Match an event to possible session windows using username and time

def find_session_candidates(username, event_time):
    if pd.isna(username) or pd.isna(event_time):
        return []

    # Normalise the event username
    username = str(username).casefold().rsplit("\\", 1)[-1]

    # Ensure the timestamp is in UTC
    event_time = pd.to_datetime(event_time, utc=True)

    # Find sessions belonging to this user
    # whose observation windows contain the event
    matches = attribution_windows[
        (attribution_windows["NormalizedUsername"] == username)
        & (attribution_windows["ObservationStartUTC"] <= event_time)
        & (attribution_windows["ObservationEndUTC"] >= event_time)
    ]

    return matches["SessionID"].tolist()

In [ ]:
# Test session-window matching using a known session

example = attribution_windows.iloc[1]

test_time = (
    example["ObservationStartUTC"]
    + pd.Timedelta(seconds=3)
)

candidates = find_session_candidates(
    example["NormalizedUsername"],
    test_time
)

print("Username:", example["NormalizedUsername"])
print("Test timestamp:", test_time)
print("Candidate sessions:", candidates)
print("Number of candidates:", len(candidates))

In [ ]:
import csv
from bisect import bisect_right
from collections import defaultdict

OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs"

# Input and output files
input_file = OUTPUT_DIR / "wazuh_identity_events.csv"
attributed_file = OUTPUT_DIR / "wazuh_attributed_events.csv"

# Organise session windows by username
windows_by_user = defaultdict(list)

for row in attribution_windows.itertuples(index=False):
    if pd.isna(row.ObservationStartUTC) or pd.isna(row.ObservationEndUTC):
        continue

    windows_by_user[row.NormalizedUsername].append({
        "SessionID": row.SessionID,
        "Start": row.ObservationStartUTC,
        "End": row.ObservationEndUTC,
        "Cutoff": row.PredictionCutoffUTC
    })

for username in windows_by_user:
    windows_by_user[username].sort(key=lambda x: x["Start"])

# Prepare efficient timestamp lookup
starts_by_user = {
    user: [w["Start"] for w in windows]
    for user, windows in windows_by_user.items()
}

max_span_by_user = {
    user: max(w["End"] - w["Start"] for w in windows)
    for user, windows in windows_by_user.items()
}

def match_event_to_session(username, event_time, available_time):
    windows = windows_by_user.get(username, [])

    if not windows or pd.isna(event_time) or pd.isna(available_time):
        return [], "missing_identity_or_time"

    starts = starts_by_user[username]
    span = max_span_by_user[username]

    right = bisect_right(starts, event_time)
    left = bisect_right(starts, event_time - span)

    candidates = [
    w for w in windows[max(0, left - 1):right]
    if (
        w["Start"] <= event_time <= w["End"]
        and w["Start"] + pd.Timedelta(seconds=2) <= available_time
    )
]

    if not candidates:
        return [], "outside_window"

    # Event must have arrived by the session's prediction cutoff
    eligible = [
        w for w in candidates
        if available_time <= w["Cutoff"]
    ]

    if not eligible:
        return [], "late_telemetry"

    if len(candidates) > 1:
        return [], "ambiguous"

    return [eligible[0]["SessionID"]], "attributed"


# Process the event dataset in chunks
stats = defaultdict(int)

with open(attributed_file, "w", newline="", encoding="utf-8") as output:
    writer = None

    for chunk in pd.read_csv(
        input_file,
        chunksize=50000,
        dtype=str
    ):

        chunk["EventTimeUTC"] = pd.to_datetime(
            chunk["WindowsEventTime"],
            utc=True,
            errors="coerce"
        )

        chunk["AvailableTimeUTC"] = pd.to_datetime(
            chunk["WazuhTimestamp"],
            utc=True,
            errors="coerce"
        )

        for record in chunk.to_dict("records"):
            identity = record.get("EventIdentity")
            username = record.get("NormalizedEventUsername")

            # Require an explicit university-domain account.
            # Bare usernames are not sufficient identity evidence.
            if not isinstance(identity, str) or not (
                any(identity.casefold().startswith(domain.casefold() + "\\")
                    for domain in LAB_CONFIG["account_domains"])
            ):
                stats["unverified_identity"] += 1
                continue

            session_ids, outcome = match_event_to_session(
                username,
                record["EventTimeUTC"],
                record["AvailableTimeUTC"]
            )

            stats[outcome] += 1

            if not session_ids:
                continue

            record["SessionID"] = session_ids[0]

            # Datetime objects need to be serialised for CSV output
            record["EventTimeUTC"] = record["EventTimeUTC"].isoformat()
            record["AvailableTimeUTC"] = record["AvailableTimeUTC"].isoformat()

            if writer is None:
                writer = csv.DictWriter(
                    output,
                    fieldnames=record.keys()
                )
                writer.writeheader()

            writer.writerow(record)

print("Attribution results:")
for status, count in stats.items():
    print(f"{status}: {count}")

print("\nSaved to:", attributed_file)

In [ ]:
import re

def normalize_windows_identity(identity):
    if not isinstance(identity, str):
        return ""

    identity = identity.strip().casefold()
    identity = re.sub(r"\\+", r"\\", identity)

    return identity

In [ ]:
# Build timestamped identity evidence from Windows telemetry

from collections import defaultdict
import pandas as pd

identity_file = OUTPUT_DIR / "wazuh_identity_events.csv"

simulation_users = set(
    attribution_windows["NormalizedUsername"].dropna()
)

# First time each domain was observed for a simulation user
domain_first_seen = defaultdict(dict)

# First time each domain/SID combination was observed
sid_first_seen = defaultdict(lambda: defaultdict(dict))

for chunk in pd.read_csv(
    identity_file,
    chunksize=50000,
    dtype=str,
    usecols=[
        "Provider",
        "EventID",
        "EventIdentity",
        "NormalizedEventUsername",
        "TargetUserSid",
        "SubjectUserSid",
        "Status",
        "WazuhTimestamp",
        "WindowsEventTime"
    ]
):
    for record in chunk.to_dict("records"):

        identity = normalize_windows_identity(
            record["EventIdentity"]
        )

        # Extract username from the normalised identity
        user = identity.rsplit("\\", 1)[-1]

        if (
            user not in simulation_users
            or "\\" not in identity
        ):
            continue

        # Parse Windows event timestamp
        event_time = pd.to_datetime(
            record["WindowsEventTime"],
            utc=True,
            errors="coerce"
        )

        # Parse Wazuh receipt timestamp
        receipt_time = pd.to_datetime(
            record["WazuhTimestamp"],
            utc=True,
            errors="coerce"
        )

        # Skip events with missing or invalid timestamps
        if pd.isna(event_time) or pd.isna(receipt_time):
            continue

        # Use the later timestamp to prevent temporal leakage
        available_time = max(event_time, receipt_time)

        provider = record["Provider"]
        event_id = record["EventID"]

        # Failed authentication attempts do not establish
        # that the target account belongs to the domain
        if provider == "Microsoft-Windows-Security-Auditing":

            if event_id == "4625":
                continue

            if event_id == "4776":
                try:
                    status = int(record["Status"], 0)
                except (ValueError, TypeError):
                    continue

                if status != 0:
                    continue

        # Extract account domain
        domain = identity.rsplit("\\", 1)[0]

        # Record when this domain was first observed
        previous = domain_first_seen[user].get(domain)

        if previous is None or available_time < previous:
            domain_first_seen[user][domain] = available_time

        # Preserve positive SID evidence from Security events
        if provider == "Microsoft-Windows-Security-Auditing":

            sid = (
                record["TargetUserSid"]
                if event_id in {"4624", "4776"}
                else record["SubjectUserSid"]
            )

            if (
                isinstance(sid, str)
                and sid.startswith("S-1-5-21-")
            ):
                previous = sid_first_seen[user][domain].get(sid)

                if previous is None or available_time < previous:
                    sid_first_seen[user][domain][sid] = available_time

print("Identity evidence index created.")
print(
    "Simulated users with domain evidence:",
    len(domain_first_seen)
)

In [ ]:
def resolve_identity(identity, cutoff):
    if not isinstance(identity, str) or not identity.strip():
        return ""

    identity = normalize_windows_identity(identity)
    username = identity.rsplit("\\", 1)[-1]

    if username not in simulation_users:
        return ""

    # Use only domains known by the event's availability time
    known_domains = {
        domain
        for domain, first_seen in domain_first_seen[username].items()
        if first_seen <= cutoff
    }

    # If multiple domains exist, require positive SID evidence
    # establishing that they represent the same account
    if len(known_domains) > 1:
        sid_sets = [
            {
                sid
                for sid, first_seen in sid_first_seen[username][domain].items()
                if first_seen <= cutoff
            }
            for domain in known_domains
        ]

        if (
            any(not sids for sids in sid_sets)
            or len(set().union(*sid_sets)) != 1
        ):
            return ""

    # Reject explicit domains that weren't established by the cutoff
    if "\\" in identity:
        domain = identity.rsplit("\\", 1)[0]

        if domain not in known_domains:
            return ""

    return username

In [ ]:
# Build process evidence for Sysmon events using ProcessGuid

from collections import defaultdict
import pandas as pd

process_history = defaultdict(list)

for chunk in pd.read_csv(
    identity_file,
    chunksize=50000,
    dtype=str,
    usecols=[
        "Provider",
        "EventID",
        "Computer",
        "ProcessGuid",
        "ParentProcessGuid",
        "EventIdentity",
        "Image",
        "WindowsEventTime",
        "WazuhTimestamp"
    ]
):
    # Select Sysmon events
    sysmon = chunk[
        chunk["Provider"] == "Microsoft-Windows-Sysmon"
    ]

    for record in sysmon.to_dict("records"):

        guid = record["ProcessGuid"]
        host = record["Computer"]

        if pd.isna(guid) or pd.isna(host):
            continue

        # Parse Windows event timestamp
        event_time = pd.to_datetime(
            record["WindowsEventTime"],
            utc=True,
            errors="coerce"
        )

        # Parse Wazuh receipt timestamp
        receipt_time = pd.to_datetime(
            record["WazuhTimestamp"],
            utc=True,
            errors="coerce"
        )

        # Skip events with missing or invalid timestamps
        if pd.isna(event_time) or pd.isna(receipt_time):
            continue

        # Use the later timestamp to prevent temporal leakage
        available_time = max(event_time, receipt_time)

        # Store process evidence by computer and ProcessGuid
        process_history[
            (host.casefold(), guid.casefold())
        ].append({
            "EventID": record["EventID"],
            "EventTime": event_time,
            "AvailableTime": available_time,
            "Identity": normalize_windows_identity(
                record["EventIdentity"]
            ),
            "Image": record["Image"],
            "ParentProcessGuid": record["ParentProcessGuid"]
        })

print("Process evidence index created.")
print(
    "Unique host/process combinations:",
    len(process_history)
)

In [ ]:
# Save process evidence for use in feature engineering

import pickle

process_file = OUTPUT_DIR / "process_history.pkl"

with open(process_file, "wb") as f:
    pickle.dump(dict(process_history), f, protocol=pickle.HIGHEST_PROTOCOL)

print("Process evidence saved to:", process_file)
print("Unique processes:", len(process_history))

In [ ]:
def resolve_process_identity(record):
    host = record.get("Computer")
    guid = record.get("ProcessGuid")

    if not isinstance(host, str) or not isinstance(guid, str):
        return ""

    key = (host.casefold(), guid.casefold())
    history = process_history.get(key, [])

    if not history:
        return ""

    # Parse Windows event timestamp
    event_time = pd.to_datetime(
        record.get("WindowsEventTime"),
        utc=True,
        errors="coerce"
    )

    # Parse Wazuh receipt timestamp
    receipt_time = pd.to_datetime(
        record.get("WazuhTimestamp"),
        utc=True,
        errors="coerce"
    )

    # Reject events with missing or invalid timestamps
    if pd.isna(event_time) or pd.isna(receipt_time):
        return ""

    # Use the later timestamp as the evidence availability time
    available_time = max(event_time, receipt_time)

    # Only use process evidence available by this cutoff
    evidence = [
        item for item in history
        if item["AvailableTime"] <= available_time
    ]

    # Identify valid process creation events
    starts = {
        item["EventTime"]
        for item in evidence
        if item["EventID"] == "1"
        and item["EventTime"] <= event_time
    }

    # Resolve process identities using normalised Windows usernames
    identities = {
        normalize_windows_identity(item["Identity"])
        for item in evidence
        if isinstance(item["Identity"], str)
        and item["Identity"]
    }

    # Identify process executable images
    images = {
        item["Image"].casefold()
        for item in evidence
        if item["EventID"] == "1"
        and isinstance(item["Image"], str)
    }

    # Identify process termination events
    ends = {
        item["EventTime"]
        for item in evidence
        if item["EventID"] == "5"
    }

    # Require exactly one process start, identity and image
    if len(starts) != 1 or len(identities) != 1 or len(images) != 1:
        return ""

    # Reject events that occurred after process termination
    if ends and event_time > min(ends):
        return ""

    # Resolve the verified process owner
    return resolve_identity(
        next(iter(identities)),
        available_time
    )

In [ ]:
# Generate corrected session-attributed events

import csv
from bisect import bisect_right
from collections import defaultdict
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs"

input_file = OUTPUT_DIR / "wazuh_identity_events.csv"
attributed_file = OUTPUT_DIR / "wazuh_attributed_events.csv"

# 1. Organise session windows by username

windows_by_user = defaultdict(list)

for row in attribution_windows.itertuples(index=False):

    if pd.isna(row.ObservationStartUTC) or pd.isna(row.ObservationEndUTC):
        continue

    windows_by_user[row.NormalizedUsername].append({
        "SessionID": row.SessionID,
        "Start": row.ObservationStartUTC,
        "ActualStart": row.StartUTC,
        "End": row.ObservationEndUTC,
        "Cutoff": row.PredictionCutoffUTC
    })

starts_by_user = {}
max_span_by_user = {}

for user, windows in windows_by_user.items():

    windows.sort(key=lambda w: (w["Start"], w["SessionID"]))

    starts_by_user[user] = [
        w["Start"] for w in windows
    ]

    max_span_by_user[user] = max(
        w["End"] - w["Start"] for w in windows
    )


# 2. Match events to session windows

def match_session(username, event_time, available_time):

    windows = windows_by_user.get(username, [])

    if not windows:
        return None, "no_session"

    starts = starts_by_user[username]
    span = max_span_by_user[username]

    right = bisect_right(starts, event_time)
    left = bisect_right(starts, event_time - span)

    # Match event time and require execution to have started
    candidates = [
        w for w in windows[max(0, left - 1):right]
        if (
            w["Start"] <= event_time <= w["End"]
            and w["ActualStart"] <= available_time
        )
    ]

    if not candidates:
        return None, "outside_window"

    # Exclude events received after the prediction cutoff
    eligible = [
        w for w in candidates
        if available_time <= w["Cutoff"]
    ]

    if not eligible:
        return None, "late_telemetry"

    # Do not assign events matching multiple sessions
    if len(candidates) > 1:
        return None, "ambiguous"

    return eligible[0]["SessionID"], "attributed"


# 3. Generate attributed events

stats = defaultdict(int)

with open(
    attributed_file,
    "w",
    newline="",
    encoding="utf-8"
) as output:

    writer = None

    for chunk in pd.read_csv(
        input_file,
        chunksize=50000,
        dtype=str
    ):

        # Windows event timestamp
        chunk["EventTimeUTC"] = pd.to_datetime(
            chunk["WindowsEventTime"],
            utc=True,
            errors="coerce"
        )

        # Wazuh receipt timestamp
        chunk["ReceiptTimeUTC"] = pd.to_datetime(
            chunk["WazuhTimestamp"],
            utc=True,
            errors="coerce"
        )

        # Availability is the later of the two timestamps
        chunk["AvailableTimeUTC"] = chunk[
            ["EventTimeUTC", "ReceiptTimeUTC"]
        ].max(axis=1)

        for record in chunk.to_dict("records"):

            event_time = record["EventTimeUTC"]
            receipt_time = record["ReceiptTimeUTC"]
            available_time = record["AvailableTimeUTC"]

            if (
                pd.isna(event_time)
                or pd.isna(receipt_time)
                or pd.isna(available_time)
            ):
                stats["missing_timestamp"] += 1
                continue

            provider = record["Provider"]
            channel = record["EventChannel"]

            # Use the same provider/channel policy as Notebook 03
            if provider == "Microsoft-Windows-Security-Auditing":
                supported = channel == "Security"

            elif provider == "Microsoft-Windows-Sysmon":
                supported = (
                    channel == "Microsoft-Windows-Sysmon/Operational"
                )

            else:
                supported = False

            if not supported:
                stats["unsupported_event"] += 1
                continue

            host = record.get("Computer")

            if not isinstance(host, str) or not host.strip():
                stats["missing_host"] += 1
                continue

            # 4. Resolve event identity

            identity = normalize_windows_identity(
                record.get("EventIdentity")
            )

            if identity and identity != "-":

                username = resolve_identity(
                    identity,
                    available_time
                )

                identity_source = "direct"

            elif provider == "Microsoft-Windows-Sysmon":

                username = resolve_process_identity(record)

                identity_source = "host_process_guid"

            else:

                username = ""
                identity_source = "unresolved"

            if not username:
                stats["unresolved_identity"] += 1
                continue

            # 5. Match event to its session

            session_id, outcome = match_session(
                username,
                event_time,
                available_time
            )

            stats[outcome] += 1

            if session_id is None:
                continue

            # Record the verified attribution
            record["SessionID"] = session_id
            record["ResolvedUsername"] = username
            record["IdentitySource"] = identity_source

            # Convert timestamps for CSV output
            record["EventTimeUTC"] = event_time.isoformat()
            record["ReceiptTimeUTC"] = receipt_time.isoformat()
            record["AvailableTimeUTC"] = available_time.isoformat()

            if writer is None:

                writer = csv.DictWriter(
                    output,
                    fieldnames=list(record.keys())
                )

                writer.writeheader()

            writer.writerow(record)


# 6. Display attribution results

print("Corrected attribution results:")

for outcome, count in sorted(stats.items()):
    print(f"{outcome}: {count:,}")

print("\nSaved to:", attributed_file)

In [ ]:
# Identify and exclude simulation orchestration artefacts

import pandas as pd
import re
import ntpath

OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs"

input_file = OUTPUT_DIR / "wazuh_attributed_events.csv"
audit_file = OUTPUT_DIR / "wazuh_attribution_audit.csv"
filtered_file = OUTPUT_DIR / "wazuh_attributed_events_filtered.csv"

ARTIFACT_COMPONENTS = {
    "m05queue",
    "m05results",
    "schedulerstate",
    "dataset2026"
}

# Load the session-attributed events
events = pd.read_csv(input_file, dtype=str).fillna("")


def get_resource_path(row):
    provider = row["Provider"]
    event_id = row["EventID"]

    if (
        provider == "Microsoft-Windows-Security-Auditing"
        and event_id == "5145"
    ):
        base = row["ShareLocalPath"] or row["ShareName"]
        relative = row["RelativeTargetName"]

        if base and relative:
            return ntpath.normpath(
                ntpath.join(base, relative.lstrip("\\/"))
            )

        return ""

    if (
        provider == "Microsoft-Windows-Security-Auditing"
        and event_id == "4663"
        and row["ObjectType"].casefold() == "file"
    ):
        return row["ObjectName"]

    if (
        provider == "Microsoft-Windows-Sysmon"
        and event_id in {"11", "23", "26"}
    ):
        return row["TargetFilename"]

    return ""


def is_simulation_artifact(row):
    paths = [
        get_resource_path(row),
        row["TargetFilename"],
        row["ObjectName"],
        row["ShareName"]
    ]

    for path in paths:
        if not path:
            continue

        components = re.split(r"[\\/]", path.casefold())

        if any(
            component in ARTIFACT_COMPONENTS
            for component in components
        ):
            return True

    return False


# Identify orchestration artefacts
events["IsSimulationArtifact"] = events.apply(
    is_simulation_artifact,
    axis=1
)

# Preserve all attributed events for auditing
events.to_csv(audit_file, index=False)

# Exclude orchestration artefacts from feature calculations
filtered_events = events.loc[
    ~events["IsSimulationArtifact"]
].drop(columns=["IsSimulationArtifact"])

filtered_events.to_csv(filtered_file, index=False)

print("Total attributed events:", len(events))
print(
    "Orchestration artefacts:",
    int(events["IsSimulationArtifact"].sum())
)
print("Events retained:", len(filtered_events))
print("\nFiltered dataset saved to:", filtered_file)

In [ ]:
# Export session prediction cutoffs for feature engineering

cutoff_file = OUTPUT_DIR / "session_prediction_cutoffs.csv"

attribution_windows[
    ["SessionID", "PredictionCutoffUTC"]
].to_csv(cutoff_file, index=False)

print("Session cutoffs exported:", len(attribution_windows))
print("Saved to:", cutoff_file)